# KFP チャレンジラボ 3

このチャレンジラボでは、チャレンジラボ 2 で開発したパイプラインを拡張してデータパイプラインを追加します。

**学習目標:**
1. Python 軽量コンポーネントを一から構築する方法を学ぶ
1. 複数のコンポーネントでパイプラインを構成する方法を学ぶ
1. Artifact パラメータにアクセスして使用する方法を学ぶ

## セットアップ

In [ ]:
from datetime import datetime

from google.cloud import aiplatform

In [ ]:
REGION = "us-central1"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]

### トレーナーイメージのビルド

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image_covertype"
IMAGE_TAG = "latest"
TRAINING_CONTAINER_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"
TRAINING_CONTAINER_IMAGE_URI

このノートブックでは、トレーニングコンテナが `us-docker.pkg.dev/{PROJECT_ID}/asl-artifact-repo/trainer_image_covertype` の下にすでに作成されていることを前提としています。以下のコマンドで存在するかどうかを確認できます。

In [ ]:
!gcloud artifacts docker images list \
--include-tags us-docker.pkg.dev/$PROJECT_ID/$ARTIFACT_REGISTRY_DIR/$IMAGE_NAME \
--filter TAGS:$IMAGE_TAG

イメージが存在しない場合は、以下のコメントアウトを解除してビルドしてください。

In [ ]:
# !gcloud builds submit --timeout 15m --tag $TRAINING_CONTAINER_IMAGE_URI trainer_image

モデルのサービング時にトレーニング時に使用した ML フレームワークのバージョンと一致させるために、以下のサービングコンテナをパイプラインに提供する必要があります:

In [ ]:
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
)

**注:** トレーニング ML フレームワークのバージョンを変更する場合は、一致するバージョンのサービングコンテナを提供する必要があります（[予測用の事前構築済みコンテナ](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers) を参照）。

## チャレンジラボ 3

このラボでは、[challenge_lab_2.ipynb](./challenge_lab_2.ipynb) で開発したパイプラインを拡張し、[ウォークスルーノートブック](../../walkthrough/solutions/kfp_walkthrough.ipynb) で行ったようにデータセットの作成とエクスポートを自動化します。<br>
これまでのパイプラインはデータセットがすでに作成されていることを前提としていましたが、ここではこの機能を追加して完全なパイプラインを完成させます。

この機能を追加するために、[`pipeline/extract_bq.py`](./pipeline/extract_bq.py) と [`pipeline/pipeline.py`](./pipeline/pipeline.py) を変更して、パイプラインに次の 2 つの要素を追加します。

- **データ分割の作成（トレーニング用と検証用の両方）**
  - [事前構築済みの BigqueryQueryJobOp コンポーネント](https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-2.8.0/api/v1/bigquery.html?h=bigqueryqueryjobop#) を使用します。
    - データ分割クエリについては、使用した [ウォークスルーノートブック](../../walkthrough/solutions/kfp_walkthrough.ipynb) を参照してください。
    - 出力テーブルの作成には、`SELECT` 句に `CREATE TABLE` 句を単純に追加するか、`destination_table` 引数を使用して出力を指定できます（この場合、引数に [google.BQTable Artifact](https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-2.8.0/api/artifact_types.html#google_cloud_pipeline_components.types.artifact_types.BQTable) を渡す必要があることに注意してください）。
  - このコンポーネントは、出力テーブル情報を含む [google.BQTable Artifact](https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-2.8.0/api/artifact_types.html#google_cloud_pipeline_components.types.artifact_types.BQTable) を返すことに注意してください。
- **BQ から GCS へのデータのエクスポート（トレーニング用と検証用の両方）**
  - [`pipeline/extract_bq.py`](./pipeline/extract_bq.py) を開き、Python 軽量コンポーネントを定義します。
  - データ抽出には Python SDK API を使用します。サンプルコードについては [公式ドキュメント](https://cloud.google.com/bigquery/docs/exporting-data#python) を参照してください。
    - [BQ クライアント](https://cloud.google.com/python/docs/reference/bigquery/latest/google.cloud.bigquery.client.Client) を作成する際はプロジェクト ID を指定してください（`client = bigquery.Client(project=<PROJECT ID>)`）。
  - このコンポーネントは `Input[BQTable]` を受け取る必要があります。
    - `artifact.metadata["parameter_name"]`（例: `bq_table_artifact.metadata["tableId"]`）を使用してアーティファクト固有のパラメータにアクセスできます。パラメータの定義についてはドキュメントページを参照してください。


[`pipeline/extract_bq.py`](./pipeline/extract_bq.py) と [`pipeline/pipeline.py`](./pipeline/pipeline.py) を開き、上記の手順に従って更新してください。

**ヒント: `TODO 3` を検索して、更新が必要なセクションを見つけてください。**

### 参考資料:
- Python 軽量コンポーネント: https://www.kubeflow.org/docs/components/pipelines/v2/components/lightweight-python-components/
- KFP でのアーティファクト出力の追加: https://www.kubeflow.org/docs/components/pipelines/v2/data-types/artifacts/#traditional-artifact-syntax 
- 事前構築済み BigqueryQueryJobOp コンポーネント: https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-2.8.0/api/v1/bigquery.html?h=bigqueryqueryjobop#
- BigQuery データエクスポート用 Python SDK: https://cloud.google.com/bigquery/docs/exporting-data#python


### 期待される結果
KFP パイプライン DAG（強調表示されたセクションを拡張）:

<img src="https://github.com/GoogleCloudPlatform/asl-ml-immersion/assets/6895245/f34bcd64-fc4e-4808-a96d-2387d54c86f0" width="500"/>

## パイプラインのコンパイルと実行

まず、パイプラインコンパイラに渡される環境変数を定義しましょう:

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATA_ROOT = f"{ARTIFACT_STORE}/data"

TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
BASE_OUTPUT_DIR = f"{ARTIFACT_STORE}/models/{TIMESTAMP}"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT_ID={PROJECT_ID}
%env REGION={REGION}
%env SERVING_CONTAINER_IMAGE_URI={SERVING_CONTAINER_IMAGE_URI}
%env TRAINING_CONTAINER_IMAGE_URI={TRAINING_CONTAINER_IMAGE_URI}
%env TRAINING_FILE_PATH={TRAINING_FILE_PATH}
%env VALIDATION_FILE_PATH={VALIDATION_FILE_PATH}
%env BASE_OUTPUT_DIR={BASE_OUTPUT_DIR}

`ARTIFACT_STORE` が作成されていることを確認し、作成されていない場合は作成しましょう:

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

**注:** アーティファクトストアが事前に作成および適切に設定されていない場合、Agent Platform からのアクセスを許可するために **Cloud Shell** で次のコマンドを実行する必要がある場合があります:

```
PROJECT_ID=$(gcloud config get-value project)
PROJECT_NUMBER=$(gcloud projects list --filter="name=$PROJECT_ID" --format="value(PROJECT_NUMBER)")
gcloud projects add-iam-policy-binding $PROJECT_ID     --member="serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com"     --role="roles/storage.objectAdmin"
```

#### CLI コンパイラを使用してパイプラインをコンパイルする

次のコマンドを使用して、生成した Python ファイルからパイプラインを YAML 記述にコンパイルします:

In [ ]:
PIPELINE_YAML = "covertype_kfp_pipeline.yaml"

In [ ]:
!kfp dsl compile --py pipeline/pipeline.py --output $PIPELINE_YAML

**注:** Python SDK を使用してパイプラインをコンパイルすることもできます:

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)

```

結果としてパイプラインファイルが生成されます。

In [ ]:
!head {PIPELINE_YAML}

### パイプラインパッケージのデプロイと実行

In [ ]:
EXPERIMENT_NAME = "kfp-covertype-experiment"

In [ ]:
aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    experiment=EXPERIMENT_NAME,
    experiment_tensorboard=False,
)

pipeline = aiplatform.PipelineJob(
    display_name="covertype_kfp_pipeline_challenge_lab",
    template_path=PIPELINE_YAML,
    enable_caching=True,
)

pipeline.submit(experiment=EXPERIMENT_NAME)

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.